In [2]:
!pip install -q google-generativeai pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 70.5 MB/s eta 0:00:00


In [3]:
import google.generativeai as genai
import json
import re

from google.colab import userdata, files
from typing import TypedDict, List, Optional
import pdfplumber
import io

API_KEY = userdata.get("GEMINI_API_KEYs")
genai.configure(api_key=API_KEY)

In [4]:
class Education(TypedDict):
    degree: Optional[str]
    institute: Optional[str]
    year: Optional[str]
    score: Optional[str]

class Experience(TypedDict):
    company: Optional[str]
    role: Optional[str]
    duration: Optional[str]
    highlights: List[str]

class Project(TypedDict):
    name: Optional[str]
    description: Optional[str]
    tech_stack: List[str]

class Resume(TypedDict):
    name: Optional[str]
    email: Optional[str]
    phone: Optional[str]
    linkedin_url: Optional[str]
    education: List[Education]
    skills: List[str]
    experience: List[Experience]
    projects: List[Project]
    total_experience_years: float
    summary: Optional[str]

sys_instr = """
You are an advanced AI Resume Parser.
RULES:
1. Extract ONLY information present in the resume.
2. NEVER hallucinate or invent values.
3. If a field is missing, use null.
4. Return valid JSON only.
5. Normalize Indian phone numbers to +91XXXXXXXXXX when possible.
6. Skills must be clean and without duplicates.
7. Summary must be exactly one sentence.
8. Resume may be in English or any Indian language.
9. Output values must always be in English.
10. Keep score, phone, linkedin_url as strings.
11. Projects must contain:
    - name
    - description
    - tech_stack
12. Experience must contain:
    - company
    - role
    - duration
    - highlights
13. Education must contain:
    - degree
    - institute
    - year
    - score
"""

def normalize_phone(phone):
    if not phone:
        return None

    digits = re.sub(r"\D", "", phone)
    if len(digits) == 10:
        return "+91" + digits

    if len(digits) == 12 and digits.startswith("91"):
        return "+" + digits
    return phone

def validate_email(email):
    if email is None:
        return True
    return "@" in email

def validate_year(year):
    if year is None:
        return True
    try:
        y = int(year)
        return 1990 <= y <= 2026
    except:
        return False

def validate_experience(exp):
    try:
        return float(exp) >= 0
    except:
        return False


# Method 1 - Prompt-only

In [5]:
def parse_resume_prompt_only(text):
    local_model = genai.GenerativeModel(
        model_name = "gemini-2.5-flash-lite",
        system_instruction = sys_instr
    )
    prompt = f"""
    {sys_instr}
    Parse this resume and return the output as a JSON object, and only a JSON object.
    {text}
    """
    response = local_model.generate_content(prompt)
    try:
        data = json.loads(response.text)
    except:
        print("JSON parsing failed. Retrying once...")
        retry_response = local_model.generate_content(prompt)
        try:
            data = json.loads(retry_response.text)
        except:
            print("JSON parsing failed again after retry. Returning empty data.")
            data = {}
    return post_process_resume(data)

# Method 2 - MIME Type

In [6]:
def parse_resume_mime(text):
    model_mime = genai.GenerativeModel(
        model_name = "gemini-2.5-flash-lite",
        system_instruction = sys_instr
    )
    response = model_mime.generate_content(
        text,
        generation_config = {
            "temperature": 0.2,
            "response_mime_type": "application/json",
            "max_output_tokens": 2048
        }
    )
    try:
        data = json.loads(response.text)
    except:
        print("JSON parsing failed. Retrying once...")
        response = model_mime.generate_content(
            text,
            generation_config = {
                "temperature": 0.2,
                "response_mime_type": "application/json",
                "max_output_tokens": 2048
            }
        )
        data = json.loads(response.text)
    return post_process_resume(data)

Method 3 - SCHEMA based

In [7]:
def parse_resume_schema(text):
    model_schema = genai.GenerativeModel(
        model_name = "gemini-2.5-flash-lite",
        system_instruction = sys_instr
    )
    response = model_schema.generate_content(
        text,
        generation_config={
            "temperature": 0.2,
            "response_mime_type": "application/json",
            "response_schema": Resume,
            "max_output_tokens": 2048
        }
    )
    try:
        data = json.loads(response.text)
    except:
        print("JSON parsing failed. Retrying once...")
        response = model_schema.generate_content(
            text,
            generation_config={
                "temperature": 0.2,
                "response_mime_type": "application/json",
                "response_schema": Resume,
                "max_output_tokens": 2048
            }
        )
        data = json.loads(response.text)
    return post_process_resume(data)

In [8]:
def post_process_resume(data):
    data["phone"] = normalize_phone(
        data.get("phone")
    )
    if not validate_email(data.get("email")):
        data["email"] = None
    for edu in data.get("education", []):
        if not validate_year(edu.get("year")):
            edu["year"] = None
    if not validate_experience(data.get("total_experience_years", 0)):
        data["total_experience_years"] = 0.0
    return data

# Tests

In [9]:
#Test 1
resume_1 = """
ANJALI SHARMA

Email: anjali.sharma@gmail.com | Phone: 9876543210

LinkedIn: linkedin.com/in/anjali-sharma

EDUCATION

- B.Tech in Computer Science, IIT Delhi, 2024 (CGPA: 8.9)

- 12th CBSE, DPS Noida, 2020 (94%)

SKILLS

Python, Django, FastAPI, React.js, MongoDB, PostgreSQL,
Docker, Git, AWS (basics), TensorFlow

EXPERIENCE

Software Engineering Intern, Flipkart (May 2023 - July 2023)

- Built a recommendation microservice using Python + Redis

- Reduced API latency by 35%

Open Source Contributor, scikit-learn (2022 - present)

- Merged 4 PRs related to model serialization

PROJECTS

- AgriBot: WhatsApp chatbot for farmers using Gemini API + Twilio

- StudyMate: Note-summarizer Chrome extension (1.2k users)
"""

#Test 2
resume_2 = """
hii i am ravi (ravi.k@yahoo.in / 8765-432-100). recently finished
my btech from vit vellore (2023, cgpa around 8). i know java/python/c++
and a bit of ml stuff. did an internship at zoho for 6 months as software
trainee. also worked at a startup called bytemate for 8 months as backend
developer where i built their payment gateway. love football and travelling.
also made a small project called NoteShare - a notes sharing app for
college students built with react and firebase.
"""


print("=" * 60)
print("TEST RESUME 1")
print("=" * 60)
print()
print("=" * 60)
print("METHOD 1 — PROMPT ONLY")
print("=" * 60)
parsed_1 = parse_resume_prompt_only(resume_1)
print(json.dumps(parsed_1, indent=2, ensure_ascii=False))
print("\nSUMMARY:")
name = parsed_1.get('name') if isinstance(parsed_1, dict) else 'N/A'
email = parsed_1.get('email') if isinstance(parsed_1, dict) else 'N/A'
phone = parsed_1.get('phone') if isinstance(parsed_1, dict) else 'N/A'
skills_list = parsed_1.get('skills') if isinstance(parsed_1, dict) and parsed_1.get('skills') else []
skills_summary = skills_list[:3] if skills_list else 'N/A'
total_experience_years = parsed_1.get('total_experience_years') if isinstance(parsed_1, dict) and parsed_1.get('total_experience_years') is not None else 'N/A'

print(
    f"Name: {name} | "
    f"Email: {email} | "
    f"Phone: {phone} | "
    f"Skills: {skills_summary} | "
    f"Experience: {total_experience_years} years"
)
print("\n\n")

print("=" * 60)
print("METHOD 2 — MIME TYPE")
print("=" * 60)
parsed_2 = parse_resume_mime(resume_1)
print(json.dumps(parsed_2, indent=2, ensure_ascii=False))
print("\nSUMMARY:")
name_2 = parsed_2.get('name') if isinstance(parsed_2, dict) else 'N/A'
email_2 = parsed_2.get('email') if isinstance(parsed_2, dict) else 'N/A'
phone_2 = parsed_2.get('phone') if isinstance(parsed_2, dict) else 'N/A'
skills_list_2 = parsed_2.get('skills') if isinstance(parsed_2, dict) and parsed_2.get('skills') else []
skills_summary_2 = skills_list_2[:3] if skills_list_2 else 'N/A'
total_experience_years_2 = parsed_2.get('total_experience_years') if isinstance(parsed_2, dict) and parsed_2.get('total_experience_years') is not None else 'N/A'
print(
    f"Name: {name_2} | "
    f"Email: {email_2} | "
    f"Phone: {phone_2} | "
    f"Skills: {skills_summary_2} | "
    f"Experience: {total_experience_years_2} years"
)
print("\n\n")

print("=" * 60)
print("METHOD 3 — SCHEMA")
print("=" * 60)
parsed_3 = parse_resume_schema(resume_1)
print(json.dumps(parsed_3, indent=2, ensure_ascii=False))

print("\nSUMMARY:")
name_3 = parsed_3.get('name') if isinstance(parsed_3, dict) else 'N/A'
email_3 = parsed_3.get('email') if isinstance(parsed_3, dict) else 'N/A'
phone_3 = parsed_3.get('phone') if isinstance(parsed_3, dict) else 'N/A'
skills_list_3 = parsed_3.get('skills') if isinstance(parsed_3, dict) and parsed_3.get('skills') else []
skills_summary_3 = skills_list_3[:3] if skills_list_3 else 'N/A'
total_experience_years_3 = parsed_3.get('total_experience_years') if isinstance(parsed_3, dict) and parsed_3.get('total_experience_years') is not None else 'N/A'
print(
    f"Name: {name_3} | "
    f"Email: {email_3} | "
    f"Phone: {phone_3} | "
    f"Skills: {skills_summary_3} | "
    f"Experience: {total_experience_years_3} years"
)

print("\n\n")
print("=" * 60)
print("TEST RESUME 2")
print("=" * 60)
parsed_test2 = parse_resume_schema(resume_2)
print(json.dumps(parsed_test2, indent=2, ensure_ascii=False))

TEST RESUME 1

METHOD 1 — PROMPT ONLY
JSON parsing failed. Retrying once...
JSON parsing failed again after retry. Returning empty data.
{
  "phone": null
}

SUMMARY:
Name: None | Email: None | Phone: None | Skills: N/A | Experience: N/A years



METHOD 2 — MIME TYPE
{
  "name": "ANJALI SHARMA",
  "email": "anjali.sharma@gmail.com",
  "phone": "+919876543210",
  "linkedin_url": "https://linkedin.com/in/anjali-sharma",
  "summary": "B.Tech student with skills in Python, Django, FastAPI, React.js, MongoDB, PostgreSQL, Docker, Git, AWS, and TensorFlow, seeking to leverage experience in software engineering and open-source contributions.",
  "education": [
    {
      "degree": "B.Tech in Computer Science",
      "institute": "IIT Delhi",
      "year": 2024,
      "score": "8.9"
    },
    {
      "degree": "12th CBSE",
      "institute": "DPS Noida",
      "year": 2020,
      "score": "94%"
    }
  ],
  "skills": [
    "Python",
    "Django",
    "FastAPI",
    "React.js",
    "MongoDB",


In [13]:
print("=" * 60)
print("TEST RESUME 3")
print("=" * 60)
uploaded = files.upload()
resume_text = ""
for filename in uploaded.keys():
    print(f'Uploaded file: {filename}')
    with pdfplumber.open(io.BytesIO(uploaded[filename])) as pdf:
        for page in pdf.pages:
            resume_text += page.extract_text() + "\n"

print("\n--- Extracted Resume Text ---\n")
print(resume_text)
print("\n--- Parsed Resume (using parse_resume_schema) ---\n")
parsed_resume = parse_resume_schema(resume_text)
print(json.dumps(parsed_resume, indent=2, ensure_ascii=False))

TEST RESUME 3


Saving sample_resume.pdf to sample_resume (1).pdf
Uploaded file: sample_resume (1).pdf

--- Extracted Resume Text ---

SHIV KUMAR
Email: shivkumar.dev@gmail.com
Phone: 9876543210
LinkedIn: https://linkedin.com/in/shiv-kumar
GitHub: https://github.com/shivkumar
Education
B.Tech in Computer Science Engineering - Mysore Institute of Technology (2025) | CGPA: 8.7
PUC - Science - Vidya Jyothi PU College (2021) | Percentage: 92%
Skills
Python, Java, C++, FastAPI, Flask, React.js, MongoDB, PostgreSQL, Docker, Git, AWS, TensorFlow,
Machine Learning, DSA
Experience
Software Development Intern, Infosys (June 2024 - August 2024)
- Developed REST APIs using FastAPI
- Improved database query performance by 30%
- Worked on backend microservices
Backend Developer Intern, TechNova Solutions (January 2024 - May 2024)
- Built authentication system using JWT
- Integrated MongoDB database
- Created API documentation using Swagger
Projects
AI Resume Parser — Built an AI-powered resume parser using Gemini A

In [11]:
print("=" * 60)
print("FINAL COMPARISON")
print("=" * 60)
comparison = {
    "Prompt Only": {
        "JSON Quality": "Medium",
        "Reliability": "Low",
        "Hallucination Risk": "High"
    },

    "MIME Type": {
        "JSON Quality": "Better",
        "Reliability": "Medium",
        "Hallucination Risk": "Medium"
    },

    "Schema": {
        "JSON Quality": "Best",
        "Reliability": "High",
        "Hallucination Risk": "Lowest"
    }
}
print(json.dumps(comparison, indent=2))

FINAL COMPARISON
{
  "Prompt Only": {
    "JSON Quality": "Medium",
    "Reliability": "Low",
    "Hallucination Risk": "High"
  },
  "MIME Type": {
    "JSON Quality": "Better",
    "Reliability": "Medium",
    "Hallucination Risk": "Medium"
  },
  "Schema": {
    "JSON Quality": "Best",
    "Reliability": "High",
    "Hallucination Risk": "Lowest"
  }
}
